In [ ]:
import numpy as np
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
import random
import torch
from model_utils import (
    generate_images_from_latents,
    get_text_embeddings,
    plot_thumbnails
)
from diffusers import (
    DDIMScheduler,
    StableDiffusionPipeline
)

from tqdm.auto import tqdm
from matplotlib import pyplot as plt
%matplotlib inline
from PIL import Image
from typing import Callable, List, Iterable, Optional, Tuple, Union

# Hyperparams

In [ ]:
prompt = "an espresso machine"
model_name = "stabilityai/stable-diffusion-2-1"
num_inference_steps = 50
guidance_scale = 7.5
save_dir = "test_repertoire_dataset/"+ prompt.replace(" ", "_")
os.makedirs(save_dir, exist_ok=True) 
SEED = 42

# Load the Models

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5").to("cuda")
vae = pipe.vae
tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder
unet = pipe.unet
scheduler = DDIMScheduler.from_config(pipe.scheduler.config)

In [ ]:
def show_image(image):
    image_array = np.array(image.cpu())
    image_array = (image_array - image_array.min()) / (image_array.max() - image_array.min() + 1e-8)
    image_array = (image_array * 255).astype(np.uint8)
    image_array = np.transpose(image_array, (1, 2, 0))
    image_plot = Image.fromarray(image_array, mode="RGB")

    # Show with corrected aspect ratio
    plt.figure(figsize=(10, 2))  # Adjust figsize to better match the image shape
    plt.imshow(image_plot)
    plt.axis('off')  # Optional: hides axis for better viewing
    plt.tight_layout()
    plt.show()

In [ ]:
def save_image(image, path):
    image_array = np.array(image.cpu())
    image_array = (image_array - image_array.min()) / (image_array.max() - image_array.min() + 1e-8)
    image_array = (image_array * 255).astype(np.uint8)
    image_array = np.transpose(image_array, (1, 2, 0))
    image_plot = Image.fromarray(image_array, mode="RGB")
    image_plot.save(path)

In [ ]:
init_prompt_embedding = get_text_embeddings(
    prompt=prompt, batch_size=1, tokenizer=tokenizer, text_encoder=text_encoder, with_grad=False)

generator = generator=torch.Generator(device="cuda").manual_seed(SEED)
init_latent = torch.randn(1, 4, 64, 64, device="cuda", generator=generator)
init_latent *= scheduler.init_noise_sigma

images = generate_images_from_latents(
    latents=init_latent,
    prompt_embedding=init_prompt_embedding,
    vae=vae,
    unet=unet,
    scheduler=scheduler,
    num_inference_steps=num_inference_steps,
    guidance_scale=guidance_scale,
)
show_image(images[0])

In [ ]:
save_image(images[0], os.path.join(save_dir, "initial_image.png"))

# PROMPTS

## Handcrafted prompts

In [ ]:
handcrafted_prompts = [
    "an espresso machine on a red table",
    "an espresso machine on a marble countertop",
    "an espresso machine inside a cozy kitchen with sunlight",
    "an espresso machine in a dark industrial café",
    "an espresso machine made of copper with vintage design",
    "an espresso machine surrounded by coffee beans",
    "an espresso machine in a modern minimalist kitchen",
    "an espresso machine next to a croissant and newspaper",
    "an espresso machine with steam rising dramatically",
    "an espresso machine with blue LED lights",
    "an espresso machine under neon lighting",
    "an espresso machine sitting in a rustic wooden cabin",
    "an espresso machine beside a window with raindrops",
    "an espresso machine in an Italian coffee bar",
    "an espresso machine made of transparent glass",
    "an espresso machine painted bright yellow",
    "an espresso machine covered in chrome reflections",
    "an espresso machine surrounded by latte art cups",
    "an espresso machine with a robotic arm pouring coffee",
    "an espresso machine on a vintage tiled floor",
    "an espresso machine surrounded by house plants",
    "an espresso machine next to a coffee grinder",
    "an espresso machine with milk frothing beside it",
    "an espresso machine designed like a spaceship",
    "an espresso machine shaped like a cat",
    "an espresso machine in a futuristic kitchen",
    "an espresso machine on a snowy mountain cabin counter",
    "an espresso machine glowing under candlelight",
    "an espresso machine on a messy artist’s desk",
    "an espresso machine surrounded by pastries and sweets",
    "an espresso machine made from bamboo and glass",
    "an espresso machine with floral engravings",
    "an espresso machine dripping coffee slowly",
    "an espresso machine photographed from above",
    "an espresso machine seen through a mirror reflection",
    "an espresso machine with a blue ceramic cup beneath it",
    "an espresso machine in a morning sunlit kitchen",
    "an espresso machine surrounded by spilled coffee beans",
    "an espresso machine sitting on a picnic table outdoors",
    "an espresso machine next to a laptop and notebook",
    "an espresso machine designed in Art Deco style",
    "an espresso machine on a rustic stone countertop",
    "an espresso machine with steam highlighted by sunlight",
    "an espresso machine on a reflective black surface",
    "an espresso machine in a minimalist white space",
    "an espresso machine surrounded by kitchen utensils",
    "an espresso machine in a dimly lit café",
    "an espresso machine made entirely of wood",
    "an espresso machine surrounded by glowing fairy lights",
    "an espresso machine with colorful ceramic mugs",
    "an espresso machine on a vintage wooden table",
    "an espresso machine surrounded by stainless steel tools",
    "an espresso machine on a marble island with coffee cups",
    "an espresso machine in a dark moody atmosphere",
    "an espresso machine sitting next to an open cookbook",
    "an espresso machine in a cozy cottage kitchen",
    "an espresso machine with latte art foam nearby",
    "an espresso machine against a brick wall background",
    "an espresso machine photographed with bokeh lights",
    "an espresso machine made of gold and black metal",
    "an espresso machine surrounded by books and plants",
    "an espresso machine on a kitchen counter at night",
    "an espresso machine next to a glass of water and spoon",
    "an espresso machine on a white tablecloth with pastries",
    "an espresso machine inside a coffee shop window",
    "an espresso machine on a reflective marble counter",
    "an espresso machine glowing softly in the dark",
    "an espresso machine surrounded by spilled milk foam",
    "an espresso machine on a red checkered tablecloth",
    "an espresso machine next to coffee beans and cinnamon",
    "an espresso machine sitting in a café with customers",
    "an espresso machine on a black granite surface",
    "an espresso machine under warm golden lighting",
    "an espresso machine made from transparent acrylic",
    "an espresso machine with visible internal mechanics",
    "an espresso machine surrounded by art posters",
    "an espresso machine in a high-tech kitchen environment",
    "an espresso machine covered in morning sunlight",
    "an espresso machine in a clean Scandinavian kitchen",
    "an espresso machine surrounded by digital screens",
    "an espresso machine next to a window with mountain view",
    "an espresso machine sitting on a wooden breakfast bar",
    "an espresso machine surrounded by chocolate desserts",
    "an espresso machine next to a plate of cookies",
    "an espresso machine in a vintage diner setting",
    "an espresso machine surrounded by fresh flowers",
    "an espresso machine with a mirror reflection beneath it",
    "an espresso machine on a countertop near a cat",
    "an espresso machine surrounded by steam and mist",
    "an espresso machine inside a futuristic spaceship galley",
    "an espresso machine in a luxury penthouse kitchen",
    "an espresso machine surrounded by colorful mugs",
    "an espresso machine with a rustic brick wall behind it",
    "an espresso machine beside a vase of tulips",
    "an espresso machine captured in dramatic side lighting",
    "an espresso machine with spilled espresso nearby",
    "an espresso machine sitting on a metal counter",
    "an espresso machine viewed from above in flat lay style",
    "an espresso machine photographed in macro detail",
    "an espresso machine in a warm candlelit kitchen",
    "an espresso machine with a steaming cup beside it",
    "an espresso machine made of brushed steel and copper",
    "an espresso machine surrounded by fruit and pastries",
    "an espresso machine on a kitchen island with stools",
    "an espresso machine in a modern black kitchen",
    "an espresso machine in a tiny home kitchen",
    "an espresso machine surrounded by cozy morning light",
    "an espresso machine photographed in cinematic lighting",
    "an espresso machine in a sunlit loft apartment",
    "an espresso machine surrounded by gadgets and cables",
    "an espresso machine covered in coffee splashes",
    "an espresso machine on a concrete surface",
    "an espresso machine next to a potted plant",
    "an espresso machine on a balcony overlooking a city",
    "an espresso machine inside a camper van kitchen",
    "an espresso machine surrounded by vintage decor",
    "an espresso machine made of glass and brass",
    "an espresso machine with colorful reflections",
    "an espresso machine near a chalkboard café menu",
    "an espresso machine on a white marble background",
    "an espresso machine on a dark wooden table",
    "an espresso machine surrounded by cream and sugar bowls",
    "an espresso machine beside a stack of coffee cups",
    "an espresso machine photographed in natural daylight",
    "an espresso machine with artistic blur effect",
    "an espresso machine surrounded by coffee accessories",
    "an espresso machine inside a coffee roastery",
    "an espresso machine in a minimal monochrome setting",
    "an espresso machine covered in pastel colors",
    "an espresso machine surrounded by light reflections",
    "an espresso machine with a small espresso cup beneath it",
    "an espresso machine on a kitchen shelf",
    "an espresso machine beside a steaming mug of coffee",
    "an espresso machine with a futuristic holographic display",
    "an espresso machine designed like a robot",
    "an espresso machine surrounded by spilled sugar",
    "an espresso machine with vintage brass levers",
    "an espresso machine on a table with morning sunlight",
    "an espresso machine beside an open window and breeze",
    "an espresso machine on a white kitchen island",
    "an espresso machine inside a bakery counter",
    "an espresso machine surrounded by chocolate bars",
    "an espresso machine with steam backlit dramatically",
    "an espresso machine designed in steampunk style",
    "an espresso machine in a cluttered artist studio",
    "an espresso machine made from rough concrete texture",
    "an espresso machine in a calm minimalist space",
    "an espresso machine under strong directional lighting",
    "an espresso machine surrounded by coffee tools",
    "an espresso machine on a granite kitchen top",
    "an espresso machine inside a dark coffee shop",
    "an espresso machine with glowing orange buttons",
    "an espresso machine beside a croissant and espresso cup",
    "an espresso machine surrounded by fog and bokeh lights",
    "an espresso machine in a high-end restaurant bar",
    "an espresso machine with water drops on it",
    "an espresso machine on a reflective glass table",
    "an espresso machine surrounded by coffee beans and cups",
    "an espresso machine inside a modern white kitchen",
    "an espresso machine captured in vintage film style",
    "an espresso machine with modern geometric shapes",
    "an espresso machine beside a pile of roasted beans",
    "an espresso machine sitting in a dimly lit bar",
    "an espresso machine under blue ambient lighting",
    "an espresso machine made of matte black steel",
    "an espresso machine on a barista counter",
    "an espresso machine surrounded by barista tools",
    "an espresso machine in an office break room",
    "an espresso machine next to an alarm clock",
    "an espresso machine surrounded by steam clouds",
    "an espresso machine on a kitchen table with sunlight",
    "an espresso machine viewed from a low angle",
    "an espresso machine surrounded by reflections",
    "an espresso machine next to spilled coffee grounds",
    "an espresso machine on a marble counter with croissants",
    "an espresso machine on a breakfast tray",
    "an espresso machine inside a luxury kitchen interior",
    "an espresso machine surrounded by cups and saucers",
    "an espresso machine in a bar with neon signs",
    "an espresso machine in a home kitchen at night",
    "an espresso machine on a colorful tiled counter",
    "an espresso machine surrounded by modern decor",
    "an espresso machine made of brass and wood",
    "an espresso machine with an artistic background",
    "an espresso machine beside a hand pouring milk",
    "an espresso machine with latte art and foam",
    "an espresso machine next to an open window and plants",
    "an espresso machine surrounded by colorful fruits",
    "an espresso machine captured in soft morning light",
    "an espresso machine with steam swirling in air",
    "an espresso machine sitting on a countertop with shadows",
    "an espresso machine surrounded by barista tools and cups",
    "an espresso machine made of futuristic materials",
    "an espresso machine under studio lighting",
    "an espresso machine in a retro 1950s diner",
    "an espresso machine on a wooden counter with reflections",
    "an espresso machine beside a steaming coffee cup",
    "an espresso machine surrounded by spices and beans",
    "an espresso machine on a textured kitchen surface",
    "an espresso machine made of colorful plastic parts",
]



In [ ]:
# Sample some for visualisation
sampled_prompts = random.sample(handcrafted_prompts, 10)

for p in sampled_prompts:
    print(p)

    prompt_embedding = get_text_embeddings(
        prompt=p,
        batch_size=1,  # type: ignore
        tokenizer=tokenizer,
        text_encoder=text_encoder,
        with_grad=False,  # No gradients needed for initial image generation
    )

    image = generate_images_from_latents(
        init_latent,
        prompt_embedding,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        scheduler=scheduler,
        unet=unet,
        vae=vae,
    ).squeeze()

    show_image(image)

In [ ]:
gpt_crafted_dir = save_dir + "/gpt_crafted/"
gpt_crafted_image_dir = gpt_crafted_dir + "images/"
gpt_prompt_embedding_dir = gpt_crafted_dir + "/prompt_embeddings/"
gpt_prompt_text_dir = gpt_crafted_dir + "/prompt_texts/"

os.makedirs(gpt_crafted_dir, exist_ok=True)
os.makedirs(gpt_prompt_embedding_dir, exist_ok=True)
os.makedirs(gpt_crafted_image_dir, exist_ok=True)
os.makedirs(gpt_prompt_text_dir, exist_ok=True)

for i, p in enumerate(handcrafted_prompts):
    
    prompt_path = os.path.join(gpt_prompt_text_dir, f"prompt_{i}.txt")
    with open(prompt_path, "w", encoding="utf-8") as f:
        f.write(p)
        
    prompt_embedding = get_text_embeddings(
        prompt=p,
        batch_size=1,  # type: ignore
        tokenizer=tokenizer,
        text_encoder=text_encoder,
        with_grad=False,  # No gradients needed for initial image generation
    )
    save_path = os.path.join(gpt_prompt_embedding_dir, f"emb_{i}.pt")
    torch.save(prompt_embedding, save_path)

    image = generate_images_from_latents(
        init_latent,
        prompt_embedding,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        scheduler=scheduler,
        unet=unet,
        vae=vae,
    ).squeeze()

    filename = f"image_{i}.png"
    save_image(image, os.path.join(gpt_crafted_image_dir, filename))

## Random Noise

In [ ]:
random_noise_prompt_embeddings = []
sigma = 0.4

for _ in range(200):
    new_emb = init_prompt_embedding + sigma * torch.randn_like(init_prompt_embedding)
    random_noise_prompt_embeddings.append(new_emb)

sample_random_noise_embeddings = random.sample(random_noise_prompt_embeddings, 10)

for emb in sample_random_noise_embeddings:
    image = generate_images_from_latents(
        init_latent,
        emb,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        scheduler=scheduler,
        unet=unet,
        vae=vae,
    ).squeeze()

    show_image(image)

random_noise_prompt_embeddings = torch.stack(random_noise_prompt_embeddings, dim=0)

In [ ]:
random_noise_dir = save_dir + f"/prompt_embed_random_noise/sigma_{sigma}/"
random_noise_prompt_embedding_dir = random_noise_dir + "/prompt_embeddings/"
random_noise_prompt_image_dir = random_noise_dir + "/images/"

os.makedirs(random_noise_dir, exist_ok=True)
os.makedirs(random_noise_prompt_embedding_dir, exist_ok=True)
os.makedirs(random_noise_prompt_image_dir, exist_ok=True)


for i, emb in enumerate(random_noise_prompt_embeddings):
    
    image = generate_images_from_latents(
        init_latent,
        emb,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        scheduler=scheduler,
        unet=unet,
        vae=vae,
    ).squeeze()

    save_path = os.path.join(random_noise_prompt_embedding_dir, f"emb_{i}.pt")
    torch.save(emb, save_path)
    
    filename = f"image_{i}.png"
    save_image(image, os.path.join(random_noise_prompt_image_dir, filename))

## SVD Mutation

### Helpers

In [ ]:
def _normalize(v: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    return v / (v.norm() + eps)

def _parse_indices(sel, n_comp: int, device=None) -> torch.Tensor:
    """Parse singular-value indices (int | list[int] | slice | None) -> sorted unique LongTensor."""
    if sel is None:
        return torch.empty(0, dtype=torch.long, device=device)

    def _wrap(i: int) -> int:
        return i + n_comp if i < 0 else i

    if isinstance(sel, int):
        idx = [_wrap(sel)]
    elif isinstance(sel, slice):
        start = 0 if sel.start is None else sel.start
        stop  = n_comp if sel.stop is None else sel.stop
        step  = 1 if sel.step is None else sel.step
        idx = list(range(start, stop, step))
    elif isinstance(sel, (list, tuple, torch.Tensor)):
        idx = [_wrap(int(i)) for i in sel]
    else:
        raise ValueError(f"Unsupported selection type for select: {type(sel)}")

    idx = sorted(set([i for i in idx if 0 <= i < n_comp]))
    return torch.tensor(idx, dtype=torch.long, device=device)

def _parse_batches(batch_select, B: int, device=None) -> torch.Tensor:
    """
    batch_select:
      - 'auto' (default): if B==2 -> [1] (cond only), else [0..B-1]
      - 'all'/'both': all rows
      - int | list/tuple[int]: explicit rows (supports negatives)
      - None: selects nothing
    """
    if batch_select is None:
        return torch.empty(0, dtype=torch.long, device=device)

    if batch_select == 'auto':
        if B == 2:
            return torch.tensor([1], dtype=torch.long, device=device)
        else:
            return torch.arange(B, dtype=torch.long, device=device)

    if batch_select in ('all', 'both'):
        return torch.arange(B, dtype=torch.long, device=device)

    # explicit indices
    if isinstance(batch_select, int):
        idx = [batch_select]
    elif isinstance(batch_select, (list, tuple, torch.Tensor)):
        idx = list(batch_select)
    else:
        raise ValueError(f"Unsupported batch_select type: {type(batch_select)}")

    out = []
    for i in idx:
        i = int(i)
        if i < 0:
            i = B + i
        if 0 <= i < B:
            out.append(i)
    out = sorted(set(out))
    return torch.tensor(out, dtype=torch.long, device=device)

def _svd_decompose(X: torch.Tensor, center_tokens: bool):
    """
    X: (seq, dim)  (float32/float16/etc). Returns U,S,Vt,X_mean with X dtype.
    """
    dtype = X.dtype
    device = X.device
    if center_tokens:
        X_mean = X.mean(dim=0, keepdim=True)
        Xc = X - X_mean
    else:
        X_mean = torch.zeros((1, X.shape[1]), dtype=dtype, device=device)
        Xc = X

    # torch.linalg.svd returns U, S, Vh (i.e., V transpose)
    U, S, Vh = torch.linalg.svd(Xc, full_matrices=False)
    return U, S, Vh, X_mean

def _svd_reconstruct(U: torch.Tensor, S: torch.Tensor, Vt: torch.Tensor, X_mean: torch.Tensor):
    return (U * S) @ Vt + X_mean

# --------------------- core: single-embedding perturb ---------------------

def _svd_perturb_single(
    X: torch.Tensor,                # (seq, dim) on some device
    mode: str,
    select,                         # int | list[int] | slice | None
    alpha: float,
    center_tokens: bool,
    seed: int | None,
):
    """
    Returns X_new: (seq, dim) with same device/dtype as X.
    """
    device, dtype = X.device, X.dtype

    U, S, Vt, X_mean = _svd_decompose(X, center_tokens)
    n = S.shape[0]
    idx = _parse_indices(select, n_comp=n, device=device)

    if idx.numel() == 0:
        return X.to(dtype=torch.float32) if dtype != torch.float32 else X

    if mode == "sigma_scale":
        S_new = S.clone()
        S_new[idx] = S[idx] * (1.0 + float(alpha))
        Vt_new = Vt
    elif mode == "sigma_shift":
        S_new = S.clone()
        S_new[idx] = S[idx] + float(alpha)
        Vt_new = Vt
    elif mode == "v_mix":
        # create local RNG for determinism if seed is given
        gen = None
        if seed is not None:
            gen = torch.Generator(device=device)
            gen.manual_seed(int(seed))

        Vt_new = Vt.clone()
        for i in idx.tolist():
            vi = Vt_new[i]
            # draw random direction orthogonal to vi
            if seed is not None:
                rand = torch.randn(vi.shape, device=device, dtype=vi.dtype, generator=gen)
            else:
                rand = torch.randn_like(vi)            # project out vi component
            rand = rand - (torch.dot(rand, vi) / (torch.dot(vi, vi) + 1e-8)) * vi
            rand_dir = _normalize(rand)
            mixed   = _normalize(vi + float(alpha) * rand_dir)
            Vt_new[i] = mixed
        S_new = S
    else:
        raise ValueError(f"Unknown mode: {mode}")

    X_new = _svd_reconstruct(U, S_new, Vt_new, X_mean)
    return X_new.to(dtype=torch.float32)

# --------------------- batched API (B, seq, dim) or (seq, dim) ---------------------

def svd_perturb(
    prompt_emb: torch.Tensor,         # (seq, dim) | (B, seq, dim)  torch.Tensor
    mode: str = "sigma_scale",        # "sigma_scale" | "sigma_shift" | "v_mix"
    select=None,                      # e.g., 0, [0,3], slice(0,2)
    alpha: float = 0.1,
    center_tokens: bool = False,
    seed: int = 0,
    batch_select: Union[str, int, List[int], None] = 'auto',
) -> torch.Tensor:
    """
    If input is (seq, dim), returns (1, seq, dim) (to match CFG batch semantics).
    If input is (B, seq, dim), returns (B, seq, dim) with selected batch rows perturbed.
    """
    if not isinstance(prompt_emb, torch.Tensor):
        raise TypeError("prompt_emb must be a torch.Tensor")

    if prompt_emb.ndim == 2:
        # (seq, dim) -> treat as single row, then unsqueeze to (1, seq, dim)
        X = prompt_emb
        out = _svd_perturb_single(
            X, mode=mode, select=select, alpha=alpha,
            center_tokens=center_tokens, seed=seed
        ).unsqueeze(0)
        return out

    elif prompt_emb.ndim == 3:
        B, S, D = prompt_emb.shape
        device = prompt_emb.device
        targets = _parse_batches(batch_select, B, device=device)
        out_rows = []
        for b in range(B):
            Xb = prompt_emb[b]
            if (targets == b).any():
                Xb_new = _svd_perturb_single(
                    Xb, mode=mode, select=select, alpha=alpha,
                    center_tokens=center_tokens, seed=seed + b
                )
            else:
                Xb_new = Xb.to(dtype=torch.float32)
            out_rows.append(Xb_new)
        return torch.stack(out_rows, dim=0)

    else:
        raise ValueError(f"Unexpected embedding shape: {tuple(prompt_emb.shape)}")

### Testing

In [ ]:
alphas = [-0.5, -0.4, -0.3, -0.2, -0.1, 0.1, 0.2, 0.3, 0.4, 0.5]
svd_prompt_embeddings = []

for n in [0, 1, 2, 3, 4]:
    for alpha in alphas:
        # multiply ONLY the nth singular value on cond
        new_embed = svd_perturb(
            init_prompt_embedding, 
            mode="sigma_scale",
            select=n, 
            alpha=alpha, 
            batch_select=1
        )
        svd_prompt_embeddings.append(new_embed)    

        new_embed = svd_perturb(
            init_prompt_embedding, 
            mode="v_mix",
            select=n, 
            alpha=alpha, 
            batch_select=1
        )
        svd_prompt_embeddings.append(new_embed)

sampled_svd_embeddings = random.sample(svd_prompt_embeddings, 10)

for emb in sampled_svd_embeddings:
    image = generate_images_from_latents(
        start_latents,
        emb,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        scheduler=scheduler,
        unet=unet,
        vae=vae,
    ).squeeze()

    show_image(image)      

## TSNE Analysis

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# ---- helpers ----
def to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def pick_cond_row_if_cfg(arr):
    # arr shapes we support: (dim,), (seq, dim), (B, seq, dim)
    if arr.ndim == 3:
        B = arr.shape[0]
        if B == 2:   # assume [uncond, cond]
            arr = arr[1]   # (seq, dim)
        else:
            # if B != 2, take the first row by default
            arr = arr[0]
    return arr

def pool_tokens(arr, kind="mean"):
    # arr is (seq, dim) or (dim,)
    if arr.ndim == 1:
        return arr
    if kind == "mean":
        return arr.mean(axis=0)
    elif kind == "first":  # e.g., CLS-like
        return arr[0]
    else:
        raise ValueError("pool kind must be 'mean' or 'first'")

def embed_list_to_vectors(lst, pool_kind="mean"):
    vecs = []
    for x in lst:
        a = to_numpy(x)
        a = pick_cond_row_if_cfg(a)    # (seq, dim) or (dim,)
        v = pool_tokens(a, kind=pool_kind)  # (dim,)
        vecs.append(v.astype(np.float32))
    return np.stack(vecs, axis=0)  # (N, dim)

# ---- inputs you already have ----
# svd_prompt_embeddings: list of embeddings
# random_noise_prompt_embeddings: list of embeddings
# handcrafted_prompt_embeddings: list of embeddings
# init_prompt_embedding: single embedding

# Convert all to pooled vectors
svd_vecs   = embed_list_to_vectors(svd_prompt_embeddings, pool_kind="mean")
rnd_vecs   = embed_list_to_vectors(random_noise_prompt_embeddings, pool_kind="mean")
hand_vecs  = embed_list_to_vectors(handcrafted_prompt_embeddings, pool_kind="mean")

init_vec   = embed_list_to_vectors([init_prompt_embedding], pool_kind="mean")[0:1]  # shape (1, dim)

# Concatenate and keep labels
X = np.concatenate([svd_vecs, rnd_vecs, hand_vecs, init_vec], axis=0)
labels = (
    ["SVD"]  * len(svd_vecs) +
    ["Random"] * len(rnd_vecs) +
    ["Handcrafted"] * len(hand_vecs) +
    ["INIT"]
)

# ---- TSNE ----
# t-SNE is stochastic; set a seed for reproducibility
tsne = TSNE(n_components=2, perplexity=min(30, max(5, X.shape[0]//3)), learning_rate="auto",
            init="pca", random_state=42, n_iter=1000, verbose=0)
Y = tsne.fit_transform(X)  # (N, 2)

# ---- plot ----
plt.figure(figsize=(8, 6))

# color/marker map
color_map  = {"SVD":"tab:blue", "Random":"tab:orange", "Handcrafted":"tab:green", "INIT":"black"}
marker_map = {"SVD":"o", "Random":"s", "Handcrafted":"^", "INIT":"X"}

# plot each group separately to get a clean legend
def plot_group(name):
    idx = [i for i, lab in enumerate(labels) if lab == name]
    plt.scatter(Y[idx, 0], Y[idx, 1],
                c=color_map[name],
                marker=marker_map[name],
                label=name,
                s=40 if name != "INIT" else 80,
                edgecolors="white",
                linewidths=0.7)

for group in ["SVD", "Random", "Handcrafted", "INIT"]:
    plot_group(group)

plt.title("t-SNE of Prompt Embeddings (pooled over tokens)")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

# LATENTS

## Random Noise

In [ ]:
random_noise_latents = []
sigma = 0.6

for _ in range(200):
    new_latents = init_latent + sigma * torch.randn_like(init_latent)
    random_noise_latents.append(new_latents)

sample_random_noise_latents = random.sample(random_noise_latents, 10)

for l in sample_random_noise_latents:
    image = generate_images_from_latents(
        l,
        init_prompt_embedding,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        scheduler=scheduler,
        unet=unet,
        vae=vae,
    ).squeeze()

    show_image(image)

random_noise_latents = torch.stack(random_noise_latents, dim=0)


In [ ]:
latent_random_noise_dir = save_dir + f"/latent_random_noise/sigma_{sigma}/"
latent_random_noise_latent_dir = latent_random_noise_dir + "/latents/"
latent_random_noise_image_dir = latent_random_noise_dir + "/images/"

os.makedirs(latent_random_noise_dir, exist_ok=True)
os.makedirs(latent_random_noise_latent_dir, exist_ok=True)
os.makedirs(latent_random_noise_image_dir, exist_ok=True)


for i, lat in enumerate(random_noise_latents):
    
    image = generate_images_from_latents(
        lat,
        init_prompt_embedding,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        scheduler=scheduler,
        unet=unet,
        vae=vae,
    ).squeeze()

    save_path = os.path.join(latent_random_noise_latent_dir, f"latent_{i}.pt")
    torch.save(emb, save_path)
    
    filename = f"image_{i}.png"
    save_image(image, os.path.join(latent_random_noise_image_dir, filename))

## Changing Random Seeds

In [ ]:
random_seeds_latents = []

for _ in range(200):
    new_latents = init_latent = torch.randn(1, 4, 64, 64, device="cuda", generator=generator)
    random_seeds_latents.append(new_latents)

sample_scheduler_random_noise_latents = random.sample(random_seeds_latents, 10)

for l in sample_scheduler_random_noise_latents:
    image = generate_images_from_latents(
        l,
        init_prompt_embedding,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        scheduler=scheduler,
        unet=unet,
        vae=vae,
    ).squeeze()

    show_image(image)

In [ ]:
random_seeds_dir = save_dir + f"/latent_random_seeds/"
latent_random_seeds_latent_dir = random_seeds_dir + "/latents/"
latent_random_seeds_image_dir = random_seeds_dir + "/images/"

os.makedirs(random_seeds_dir, exist_ok=True)
os.makedirs(latent_random_seeds_latent_dir, exist_ok=True)
os.makedirs(latent_random_seeds_image_dir, exist_ok=True)


for i, lat in enumerate(random_seeds_latents):
    
    image = generate_images_from_latents(
        lat,
        init_prompt_embedding,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        scheduler=scheduler,
        unet=unet,
        vae=vae,
    ).squeeze()

    save_path = os.path.join(latent_random_seeds_latent_dir, f"latent_{i}.pt")
    torch.save(emb, save_path)
    
    filename = f"image_{i}.png"
    save_image(image, os.path.join(latent_random_seeds_image_dir, filename))

## Pullback Mutation

### Helpers

In [ ]:
def _phi_hidden(unet, x_t, t, prompt_embedding, guidance_scale, *, layer="mid_block", scheduler=None):
    """
    Capture a hidden activation from diffusers UNet via a forward hook.
    Returns the CONDITIONAL half (for CFG) flattened to 1D.
    """
    assert scheduler is not None, "Pass the scheduler so we can call scale_model_input once."

    # CFG duplication (same as your sampling path)
    uncond, cond = prompt_embedding[0:1], prompt_embedding[1:2]
    x_in = torch.cat([x_t, x_t], dim=0)          # (2,B,4,H,W) -> here B==1 so (2,4,H,W)
    t_in = torch.cat([t, t], dim=0)              # (2,)
    h_in = torch.cat([uncond, cond], dim=0)      # (2, seq, dim)

    # scale model input exactly like in your roll loop
    x_in = scheduler.scale_model_input(x_in, t)  # pass scalar timestep (broadcasts across batch)

    # pick target module
    if layer == "mid_block":
        target = unet.mid_block
    else:
        # example: ("down_blocks", -1, "resnets", -1)
        mod = unet
        for key in layer:
            mod = mod[key] if isinstance(key, int) else getattr(mod, key)
        target = mod

    captured = {}

    def _hook(_m, _inp, out):
        captured["feat"] = out

    handle = target.register_forward_hook(_hook)
    try:
        # normal UNet forward; hook grabs the chosen layer activation
        _ = unet(x_in, t_in, encoder_hidden_states=h_in)
        feats = captured.get("feat", None)
        if feats is None:
            raise RuntimeError("Hidden feature was not captured; check `layer` choice.")

        # take CONDITIONAL half (batch doubled for CFG)
        if feats.dim() >= 1 and feats.size(0) >= 2:
            feats = feats[feats.size(0)//2:]

        return feats.reshape(-1)
    finally:
        handle.remove()
        
@torch.no_grad()
def roll_to_hook(x_T, prompt_embedding, *, unet, scheduler, guidance_scale, num_steps, hook_idx, device="cuda"):
    x  = x_T.to(device).clone()
    prompt_embedding = prompt_embedding.to(device)
    scheduler.set_timesteps(num_steps)
    timesteps = scheduler.timesteps.to(device)

    for i, t in enumerate(timesteps):
        if i == hook_idx:
            return x.clone(), t, i, timesteps  # only these 4 values
        
        # Expand latents for classifier-free guidance (double batch size)
        latent_model_input = torch.cat([x] * 2)
        latent_model_input = scheduler.scale_model_input(latent_model_input, timestep=t)
        
        # Disable gradients for inference
        with torch.no_grad():
            noise_pred = unet(latent_model_input, t, encoder_hidden_states=prompt_embedding).sample

        # Apply classifier-free guidance
        noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

        # Denoising step
        x = scheduler.step(noise_pred, t, x).prev_sample
        
    # fallback if hook_idx==len-1
    return x.clone(), timesteps[-1], len(timesteps)-1, timesteps


def compute_svd_basis_at_hook(
    x_t: torch.Tensor,
    t_scalar: int,
    *,
    unet,
    scheduler,
    prompt_embedding,
    guidance_scale: float = 7.5,
    k: int = 10,
    iters: int = 20,
    fd_eps: float = 1e-3,
    device: str = "cuda",
    seed: int | None = None,
    layer="mid_block",
):
    xT = x_t.detach().to(device)
    t  = torch.tensor(t_scalar, device=device).expand(1)
    prompt_embedding = prompt_embedding.to(device)

    # --- base hidden vector: NO grad needed
    with torch.no_grad():
        y0 = _phi_hidden(
            unet, xT, t, prompt_embedding, guidance_scale,
            layer=layer, scheduler=scheduler
        )

    # --- JVP via finite-difference: NO grad needed
    def Jx_v(vx: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            y1 = _phi_hidden(
                unet, xT + fd_eps * vx, t, prompt_embedding, guidance_scale,
                layer=layer, scheduler=scheduler
            )
        return (y1 - y0) / fd_eps

    # --- VJP via autograd: GRAD NEEDED
    def JxT_u(u_flat: torch.Tensor) -> torch.Tensor:
        x_req = xT.clone().requires_grad_(True)  # make x require grad
        y = _phi_hidden(
            unet, x_req, t, prompt_embedding, guidance_scale,
            layer=layer, scheduler=scheduler
        )
        dot = (y * u_flat.detach()).sum()
        (g,) = torch.autograd.grad(
            dot, x_req, retain_graph=False, allow_unused=True
        )
        return torch.zeros_like(x_req) if g is None else g

    # --- power iteration (same as before)
    rng = None
    if seed is not None:
        rng = torch.Generator(device=device).manual_seed(int(seed))

    vecs, sigmas = [], []
    for _ in range(k):
        v = torch.randn(xT.shape, device=xT.device, generator=rng, dtype=xT.dtype)
        for p in vecs:
            v -= (v.flatten() @ p.flatten()) * p
        v = v / (v.norm() + 1e-8)

        for __ in range(iters):
            u  = Jx_v(v)         # no-grad path
            Av = JxT_u(u)        # grad-enabled path
            for p in vecs:
                Av -= (Av.flatten() @ p.flatten()) * p
            v = (Av / (Av.norm() + 1e-8)).detach()

        u  = Jx_v(v)
        Av = JxT_u(u)
        rq = (v.flatten() @ Av.flatten())
        sigmas.append(float(torch.sqrt(torch.clamp(rq, min=0.0))))
        vecs.append(v.clone())

    return vecs, sigmas

@torch.no_grad()
def apply_many_svd_edits(
    x_t: torch.Tensor,
    vecs: List[torch.Tensor],
    *,
    alpha_list: Iterable[float],
    select_list: Iterable[int | List[int]],
) -> List[torch.Tensor]:
    """
    Build multiple x_t* from a single SVD basis:
      - For each (alpha, select), form a normalized sum of chosen vecs and add alpha * step.
    Returns a list of x_t_star tensors.
    """
    x_t = x_t.detach()
    out = []

    # pre-normalize vecs we might combine
    normed = [v / (v.norm() + 1e-8) for v in vecs]
    for alpha in alpha_list:
        for sel in select_list:
            # parse selection
            if isinstance(sel, int):
                idxs = [sel]
            else:
                idxs = list(sorted(set(int(i) for i in sel)))
            if not idxs:
                out.append(x_t.clone())
                continue

            step = torch.zeros_like(normed[0])
            for i in idxs:
                step = step + normed[i]
            step = step / len(idxs)

            out.append(x_t + float(alpha) * step)

    return out

@torch.no_grad()
def invert_to_start(
        start_latents,
        prompt_embedding,
        unet,
        scheduler,
        guidance_scale,
        timesteps,
        hook_idx,
        device="cuda"
    ):
    
    scheduler.set_timesteps(len(timesteps), device="cuda")
    latents  = start_latents.clone() #* 0.18215
    timesteps = reversed(timesteps[:hook_idx + 1])
    num_reverse_steps = len(timesteps)
    
    for i in range(1, num_reverse_steps):
    
        # We'll skip the final iteration
        if i >= num_reverse_steps - 1: continue
        
        t = timesteps[i]
        
        # Expand the latents if we are doing classifier free guidance
        latent_model_input = torch.cat([latents] * 2) 
        latent_model_input = scheduler.scale_model_input(latent_model_input, t)

        # Predict the noise residual
        # Disable gradients for inference
        noise_pred = unet(latent_model_input, t, encoder_hidden_states=prompt_embedding).sample

        # Perform guidance
        noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)
        
        current_t = max(0, t.item() - (1000//scheduler.num_inference_steps)) #t
        next_t = t # min(999, t.item() + (1000//num_inference_steps)) # t+1

        alpha_t = scheduler.alphas_cumprod[current_t]
        alpha_t_next = scheduler.alphas_cumprod[next_t]
        
        # Inverted update step (re-arranging the update step to get x(t) (new latents) as a function of x(t-1) (current latents)
        latents = (latents - (1-alpha_t).sqrt()*noise_pred)*(alpha_t_next.sqrt()/alpha_t.sqrt()) + (1-alpha_t_next).sqrt()*noise_pred

    return latents  # x_T*

# ---------- 3) Batch invert: all x_t* -> x_T* ----------
@torch.no_grad()
def invert_to_start_batch(
    x_t_stars: List[torch.Tensor],
    prompt_embedding: torch.Tensor,
    *,
    unet,
    scheduler,
    guidance_scale: float,
    timesteps: torch.Tensor,
    hook_idx: int,
    device: str = "cuda",
) -> List[torch.Tensor]:
    """
    Invert many edited latents to x_T*, reusing the same (curr, next) timestep pairs.
    """
    reverted = []
    
    for x in x_t_stars:
        x_T_star = invert_to_start(
            start_latents=x,
            prompt_embedding=prompt_embedding,
            unet=unet,
            scheduler=scheduler,
            guidance_scale=guidance_scale,
            timesteps=timesteps,
            hook_idx=hook_idx,
            device=device
        )
        reverted.append(x_T_star)

    return torch.stack(reverted) 


In [ ]:
def svd_mutation_many(
    init_latent,
    prompt_embedding,
    unet,
    scheduler,
    guidance_scale: float = 7.5,
    num_inference_steps: int = 50,
    hook_proportion: float = 0.6,
    alpha_list=[0.5, 1.0],
    select_list=[0],
    op="mid",
    block_idx=0,
    device="cuda",
):
    """
    Runs one forward-to-hook, computes SVD basis at that hidden layer,
    then applies multiple edits (select×alpha) and inverts each back.
    """

    # 1. Roll forward until hook
    hook_idx = int(hook_proportion * num_inference_steps)
    x_t, t_scalar, idx, timesteps = roll_to_hook(
        x_T=init_latent,
        prompt_embedding=prompt_embedding,
        unet=unet,
        scheduler=scheduler,
        guidance_scale=guidance_scale,
        num_steps=num_inference_steps,
        hook_idx=hook_idx,
        device=device,
    )

    # 3. Compute SVD basis ONCE (heavy)
    # print(f"Computing SVD basis at layer {op}[{block_idx}]...")
    vecs, sigmas = compute_svd_basis_at_hook(
        x_t,
        t_scalar,
        unet=unet,
        scheduler=scheduler,
        prompt_embedding=prompt_embedding,
        guidance_scale=guidance_scale,
        k=10,
        iters=20,
        fd_eps=1e-3,
        device=device,
        seed=SEED,
    )
    
    # print("Top singular values:", [round(s, 4) for s in sigmas])

    # 4. Apply multiple edits (no need to recompute SVD)
    x_T_stars = []
    for sv in select_list:
        if sv >= len(vecs):
            print(f"Warning: sv={sv} exceeds computed rank={len(vecs)}")
            continue
        base_vec = vecs[sv] / (vecs[sv].norm() + 1e-8)
        for alpha in alpha_list:
            x_t_star = x_t + alpha * base_vec
            x_T_star = invert_to_start(
                x_t_star,
                prompt_embedding,
                unet=unet,
                scheduler=scheduler,
                guidance_scale=guidance_scale,
                timesteps=timesteps,
                hook_idx=idx,
                device=device,
            )
            x_T_stars.append(x_T_star)
    return x_T_stars

### Testing

In [ ]:
# large_alpha_list = [-60, -40, -20, -10, -5, -4, -3, -2, -1, -0.5, 0.5, 1, 2, 3, 4, 5, 10, 20, 40, 60]
small_alpha_list = [-1.0, -0.9, -0.8, -0.7, -0.6, -0.5, -0.4, -0.3, -0.2, -0.1, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
medium_alpha_list = [-10, -9, -8, -7, -6, -5, -4, -3, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
select_list = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
hook_proportion = 0.7

pullback_latents = svd_mutation_many(
    init_latent,
    init_prompt_embedding,
    unet,
    scheduler,
    guidance_scale=guidance_scale,
    num_inference_steps=num_inference_steps,
    hook_proportion=hook_proportion,
    alpha_list=alpha_list,
    select_list=select_list,
)

Computing SVD basis at layer mid[0]...


/tmp/ipykernel_756/2611154705.py:94: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  t  = torch.tensor(t_scalar, device=device).expand(1)


Top singular values: [107.0442, 75.372, 85.1666, 88.3001, 64.6377, 75.956, 70.4855, 56.2651, 47.9778, 51.1265]


In [84]:
pullback_latent__dir = save_dir + f"/latent_pullback/hook_proportion_{hook_proportion}/small_alphas/"
pullback_latent_latent_dir = pullback_latent__dir + "/latents/"
pullback_latents_image_dir = pullback_latent__dir + "/images/"

os.makedirs(pullback_latent__dir, exist_ok=True)
os.makedirs(pullback_latent_latent_dir, exist_ok=True)
os.makedirs(pullback_latents_image_dir, exist_ok=True)

for i, lat in enumerate(pullback_latents):
    
    image = generate_images_from_latents(
        lat,
        init_prompt_embedding,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        scheduler=scheduler,
        unet=unet,
        vae=vae,
    ).squeeze()

    save_path = os.path.join(pullback_latent_latent_dir, f"latent_{i}.pt")
    torch.save(emb, save_path)
    
    filename = f"image_{i}.png"
    save_image(image, os.path.join(pullback_latents_image_dir, filename))

## TSNE

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# ---- helpers ----
def to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def latent_to_vector(arr):
    a = to_numpy(arr)
    # common shapes: (1, 4, H, W) or (4, H, W)
    if a.ndim == 4 and a.shape[0] == 1:
        a = a[0]
    return a.astype(np.float32).ravel()  # flatten

def embed_list_to_vectors(lst):
    return np.stack([latent_to_vector(x) for x in lst], axis=0)

# ---- inputs you already have ----
# svd_latents: list of latents
# scheduler_random_noise_latents: list of latents
# random_noise_latents: list of latents
# start_latents: single latent (or list with one)

svd_vecs   = embed_list_to_vectors(svd_latents)
sched_vecs = embed_list_to_vectors(scheduler_random_noise_latents)
rand_vecs  = embed_list_to_vectors(random_noise_latents)

# make sure start_latents is a single item; wrap if needed
start_vec  = embed_list_to_vectors([start_latents])  # shape (1, D)

# concatenate + labels
X = np.concatenate([svd_vecs, sched_vecs, rand_vecs, start_vec], axis=0)
labels = (
    ["SVD"] * len(svd_vecs) +
    ["SchedulerNoise"] * len(sched_vecs) +
    ["RandomNoise"] * len(rand_vecs) +
    ["START"]
)

# ---- t-SNE ----
N = X.shape[0]
perp = min(30, max(5, N // 3))
tsne = TSNE(
    n_components=2,
    perplexity=perp,
    learning_rate="auto",
    init="pca",
    random_state=42,
    n_iter=1000,
    verbose=0,
)
Y = tsne.fit_transform(X)  # (N, 2)

# ---- plot ----
plt.figure(figsize=(8, 6))

color_map  = {"SVD":"tab:blue", "SchedulerNoise":"tab:orange", "RandomNoise":"tab:green", "START":"black"}
marker_map = {"SVD":"o",        "SchedulerNoise":"s",         "RandomNoise":"^",        "START":"X"}

def plot_group(name):
    idx = [i for i, lab in enumerate(labels) if lab == name]
    plt.scatter(
        Y[idx, 0], Y[idx, 1],
        c=color_map[name],
        marker=marker_map[name],
        label=name,
        s=40 if name != "START" else 90,
        edgecolors="white",
        linewidths=0.7,
    )

for group in ["SVD", "SchedulerNoise", "RandomNoise", "START"]:
    plot_group(group)

plt.title("t-SNE of Latents (flattened)")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.legend(frameon=True)
plt.tight_layout()
plt.show()